# Sprint 3 - Data Cleaning & Feature Engineering

## Proyecto

**Precision Agriculture Analytics for Apple Orchards**

## Objetivo

Preparar un conjunto de datos limpio, consistente y documentado para el entrenamiento de modelos de Machine Learning.

## Preguntas que responderemos

- ¿Los datos son consistentes?
- ¿Existen valores fuera de rango?
- ¿Es necesario transformar variables?
- ¿Qué nuevas variables pueden aportar información al modelo?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/raw/soil.csv")

df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [3]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Filas: 2200
Columnas: 8


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   str    
dtypes: float64(4), int64(3), str(1)
memory usage: 137.6 KB


In [5]:
missing = df.isnull().sum()

missing

N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

In [6]:
duplicates = df.duplicated().sum()

print(f"Duplicados: {duplicates}")

Duplicados: 0


In [7]:
validation = pd.DataFrame({
    "Min": df.min(numeric_only=True),
    "Max": df.max(numeric_only=True),
    "Mean": df.mean(numeric_only=True),
    "Std": df.std(numeric_only=True)
})

validation

,Min,Max,Mean,Std
N,0.000000,140.000000,50.551818,36.917334
P,5.000000,145.000000,53.362727,32.985883
K,5.000000,205.000000,48.149091,50.647931
temperature,8.825675,43.675493,25.616244,5.063749
humidity,14.258040,99.981876,71.481779,22.263812
ph,3.504752,9.935091,6.469480,0.773938
rainfall,20.211267,298.560117,103.463655,54.958389


## Validación de rangos

Se revisaron los valores mínimos, máximos, medias y desviaciones estándar de todas las variables numéricas.

El objetivo fue detectar posibles errores de captura o valores fuera de un rango agronómicamente razonable antes de construir modelos predictivos.

In [9]:
print("Valores de pH fuera del rango:", ((df["ph"] < 0) | (df["ph"] > 14)).sum())

print("Temperaturas negativas:", (df["temperature"] < 0).sum())

print("Humedad > 100:", (df["humidity"] > 100).sum())

print("N negativos:", (df["N"] < 0).sum())
print("P negativos:", (df["P"] < 0).sum())
print("K negativos:", (df["K"] < 0).sum())

Valores de pH fuera del rango: 0
Temperaturas negativas: 0
Humedad > 100: 0
N negativos: 0
P negativos: 0
K negativos: 0


In [10]:
df_ml = df.copy()

In [12]:
## Nutrientes totales
df_ml["Total_NPK"] = (
    df_ml["N"] +
    df_ml["P"] +
    df_ml["K"]
)

In [13]:
## Relaciones entre nutrientes

df_ml["N_P_ratio"] = df_ml["N"] / df_ml["P"]

df_ml["N_K_ratio"] = df_ml["N"] / df_ml["K"]

df_ml["P_K_ratio"] = df_ml["P"] / df_ml["K"]

In [14]:
## Indice exploratorio

df_ml["Soil_Fertility_Index"] = (
    df_ml["N"] +
    df_ml["P"] +
    df_ml["K"]
) / 3

In [15]:
df_ml.head()

,N,P,K,temperature,humidity,ph,rainfall,label,Total_NPK,N_P_ratio,N_K_ratio,P_K_ratio,Soil_Fertility_Index
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice,175,2.142857,2.093023,0.976744,58.333333
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice,184,1.465517,2.073171,1.414634,61.333333
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice,159,1.090909,1.363636,1.250000,53.000000
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice,149,2.114286,1.850000,0.875000,49.666667
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice,162,1.857143,1.857143,1.000000,54.000000


In [16]:
## Verificación de nuevas variables

new_features = [
    "Total_NPK",
    "N_P_ratio",
    "N_K_ratio",
    "P_K_ratio"
]

df_ml[new_features].describe()

,Total_NPK,N_P_ratio,N_K_ratio,P_K_ratio
count,2200.000000,2200.000000,2200.000000,2200.000000
mean,152.063636,1.701689,1.670539,1.668555
std,79.918669,2.573334,1.507555,1.197217
min,17.000000,0.000000,0.000000,0.090909
25%,94.000000,0.350000,0.559706,0.697874
50%,146.000000,0.890909,1.388322,1.262531
75%,179.000000,1.977399,2.167453,2.588235
max,385.000000,23.800000,9.333333,6.000000


In [17]:
## Verificación de valores infinitos

import numpy as np

df_ml.replace([np.inf, -np.inf], np.nan, inplace=True)

df_ml[new_features].isnull().sum()

Total_NPK    0
N_P_ratio    0
N_K_ratio    0
P_K_ratio    0
dtype: int64

Preparación para machine learning


In [18]:
## Variables predictoras
X = df_ml.drop("label", axis=1)

In [19]:
## Variable objetivo

y = df_ml["label"]

In [20]:
## Verificación

print("Variables predictoras:", X.shape)
print("Variable objetivo:", y.shape)

Variables predictoras: (2200, 12)
Variable objetivo: (2200,)


In [21]:
## Exportación del dataset preparado

output_path = "../data/processed/soil_ml_ready.csv"

df_ml.to_csv(output_path, index=False)

print("Dataset exportado correctamente.")

Dataset exportado correctamente.


# Conclusiones

## Calidad de los datos

El conjunto de datos presenta una estructura consistente, sin valores faltantes ni duplicados. Las variables cumplen con los rangos esperados definidos por las reglas de negocio.

## Ingeniería de características

Se incorporaron nuevas variables derivadas con el objetivo de representar relaciones entre nutrientes y resumir información relevante para los modelos de Machine Learning.

Las variables creadas fueron:

- Total_NPK
- N_P_ratio
- N_K_ratio
- P_K_ratio

La variable Soil_Fertility_Index fue evaluada durante el proceso de diseño, pero se descartó al aportar la misma información que Total_NPK.

## Preparación para Machine Learning

El dataset quedó preparado para la etapa de modelado supervisado y fue exportado a la carpeta data/processed para mantener separados los datos originales de los datos transformados.

| Verificación      | Resultado   |
| ----------------- | ----------- |
| Valores faltantes | ✅ 0         |
| Duplicados        | ✅ 0         |
| Reglas de negocio | ✅ Cumplidas |
| Variables nuevas  | ✅ 4         |
| Dataset exportado | ✅ Sí        |
